# Exercice 16 - Producer Kafka Python

## Objectifs
- Creer un producer Kafka robuste
- Envoyer des messages JSON structures
- Utiliser les cles de partitionnement
- Simuler un flux de donnees en temps reel

---

## 1. Architecture Producer

```
+------------------------------------------------------------------+
|                    PRODUCER KAFKA                                |
+------------------------------------------------------------------+
|                                                                  |
|   APPLICATION           PRODUCER              KAFKA              |
|                                                                  |
|  +-----------+      +-------------+      +----------------+      |
|  |           |      |             |      |                |      |
|  |  Donnees  |----->| Serializer  |----->|    Topic       |      |
|  |  (dict)   |      | (JSON)      |      |                |      |
|  +-----------+      +-------------+      | +------------+ |      |
|                            |             | | Partition 0| |      |
|                     +------v------+      | +------------+ |      |
|                     |             |      | | Partition 1| |      |
|                     | Partitioner |----->| +------------+ |      |
|                     | (par cle)   |      | | Partition 2| |      |
|                     +-------------+      | +------------+ |      |
|                                          |                |      |
|                                          +----------------+      |
|                                                                  |
+------------------------------------------------------------------+

Options importantes :
- bootstrap_servers : liste des brokers
- key_serializer    : serialisation des cles
- value_serializer  : serialisation des valeurs
- acks              : garantie de livraison (0, 1, all)
- retries           : nombre de tentatives en cas d'erreur
```

## 2. Configuration

In [ ]:
!pip install kafka-python -q

from kafka import KafkaProducer, KafkaAdminClient
from kafka.admin import NewTopic
import json
import time
import random
from datetime import datetime

KAFKA_BROKER = "broker:9092"

print("Configuration prete")

In [ ]:
# Creer les topics necessaires
def creer_topic_si_absent(nom, partitions=3):
    try:
        admin = KafkaAdminClient(bootstrap_servers=KAFKA_BROKER)
        if nom not in admin.list_topics():
            admin.create_topics([NewTopic(name=nom, num_partitions=partitions, replication_factor=1)])
            print(f"Topic '{nom}' cree")
        else:
            print(f"Topic '{nom}' existe deja")
        admin.close()
    except Exception as e:
        print(f"Erreur : {e}")

creer_topic_si_absent("commandes-json", 3)
creer_topic_si_absent("logs-application", 1)
creer_topic_si_absent("metrics", 3)

## 3. Producer JSON

In [ ]:
# Creer un producer avec serialisation JSON
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    acks='all',  # Attendre confirmation de tous les replicas
    retries=3,   # Reessayer 3 fois en cas d'erreur
)

print("Producer JSON configure")

In [ ]:
# Envoyer un message JSON
commande = {
    "order_id": 1001,
    "customer_id": "CUST-123",
    "timestamp": datetime.now().isoformat(),
    "produits": [
        {"product_id": 1, "quantity": 2, "price": 29.99},
        {"product_id": 5, "quantity": 1, "price": 49.99}
    ],
    "total": 109.97
}

# Utiliser customer_id comme cle (meme client = meme partition)
future = producer.send(
    topic="commandes-json",
    key=commande["customer_id"],
    value=commande
)

result = future.get(timeout=10)
print(f"Message envoye :")
print(f"  Topic: {result.topic}")
print(f"  Partition: {result.partition}")
print(f"  Offset: {result.offset}")
print(f"  Cle: {commande['customer_id']}")

## 4. Generateur de commandes aleatoires

In [ ]:
# Donnees de reference
PRODUITS = [
    {"id": 1, "nom": "Laptop", "prix": 999.99},
    {"id": 2, "nom": "Souris", "prix": 29.99},
    {"id": 3, "nom": "Clavier", "prix": 79.99},
    {"id": 4, "nom": "Ecran", "prix": 349.99},
    {"id": 5, "nom": "Casque", "prix": 149.99},
    {"id": 6, "nom": "Webcam", "prix": 89.99},
    {"id": 7, "nom": "USB Hub", "prix": 39.99},
    {"id": 8, "nom": "SSD", "prix": 129.99},
]

CLIENTS = [f"CUST-{i:03d}" for i in range(1, 21)]  # 20 clients

def generer_commande():
    """Genere une commande aleatoire"""
    client = random.choice(CLIENTS)
    nb_produits = random.randint(1, 4)
    
    produits_commande = []
    total = 0.0
    
    for _ in range(nb_produits):
        produit = random.choice(PRODUITS)
        quantite = random.randint(1, 3)
        sous_total = produit["prix"] * quantite
        
        produits_commande.append({
            "product_id": produit["id"],
            "product_name": produit["nom"],
            "quantity": quantite,
            "unit_price": produit["prix"],
            "subtotal": round(sous_total, 2)
        })
        total += sous_total
    
    return {
        "order_id": random.randint(10000, 99999),
        "customer_id": client,
        "timestamp": datetime.now().isoformat(),
        "items": produits_commande,
        "total": round(total, 2),
        "status": "created"
    }

In [ ]:
# Tester le generateur
commande_test = generer_commande()
print(json.dumps(commande_test, indent=2))

## 5. Envoi en batch

In [ ]:
# Envoyer un lot de commandes
def envoyer_batch(producer, topic, nb_messages):
    """Envoie un lot de messages"""
    futures = []
    
    for i in range(nb_messages):
        commande = generer_commande()
        future = producer.send(
            topic=topic,
            key=commande["customer_id"],
            value=commande
        )
        futures.append((commande["order_id"], future))
    
    # Attendre toutes les confirmations
    producer.flush()
    
    # Verifier les resultats
    succes = 0
    erreurs = 0
    
    for order_id, future in futures:
        try:
            result = future.get(timeout=10)
            succes += 1
        except Exception as e:
            print(f"Erreur pour commande {order_id}: {e}")
            erreurs += 1
    
    return succes, erreurs

In [ ]:
# Envoyer 50 commandes
print("Envoi de 50 commandes...")
succes, erreurs = envoyer_batch(producer, "commandes-json", 50)

print(f"\nResultat :")
print(f"  Succes : {succes}")
print(f"  Erreurs: {erreurs}")

## 6. Simulation de flux en temps reel

In [ ]:
def simuler_flux(producer, topic, nb_messages, delai_ms=500):
    """
    Simule un flux de donnees en temps reel.
    
    Args:
        producer: KafkaProducer
        topic: Nom du topic
        nb_messages: Nombre de messages a envoyer
        delai_ms: Delai entre les messages (millisecondes)
    """
    print(f"Simulation de {nb_messages} messages...")
    print(f"Delai entre messages: {delai_ms}ms")
    print("=" * 50)
    
    for i in range(nb_messages):
        commande = generer_commande()
        
        future = producer.send(
            topic=topic,
            key=commande["customer_id"],
            value=commande
        )
        
        result = future.get(timeout=10)
        
        print(f"[{i+1}/{nb_messages}] Order {commande['order_id']} | "
              f"Client {commande['customer_id']} | "
              f"Total {commande['total']:,.2f} EUR | "
              f"Partition {result.partition}")
        
        time.sleep(delai_ms / 1000.0)
    
    print("=" * 50)
    print("Simulation terminee")

In [ ]:
# Lancer une simulation (10 messages, 300ms entre chaque)
simuler_flux(producer, "commandes-json", 10, delai_ms=300)

## 7. Producer avec callbacks

In [ ]:
# Callbacks pour gerer succes et erreurs
def on_success(record_metadata):
    """Callback appele en cas de succes"""
    print(f"[OK] Topic={record_metadata.topic}, "
          f"Partition={record_metadata.partition}, "
          f"Offset={record_metadata.offset}")

def on_error(exception):
    """Callback appele en cas d'erreur"""
    print(f"[ERREUR] {exception}")

In [ ]:
# Utiliser les callbacks
for i in range(5):
    commande = generer_commande()
    
    producer.send(
        topic="commandes-json",
        key=commande["customer_id"],
        value=commande
    ).add_callback(on_success).add_errback(on_error)

producer.flush()

## 8. Producer de logs

In [ ]:
# Generateur de logs applicatifs
NIVEAUX_LOG = ["INFO", "INFO", "INFO", "WARNING", "ERROR"]  # Plus de INFO
MODULES = ["auth", "api", "database", "cache", "payment"]
MESSAGES = {
    "auth": ["User login", "User logout", "Authentication failed", "Session expired"],
    "api": ["Request received", "Response sent", "Rate limit exceeded", "Invalid request"],
    "database": ["Query executed", "Connection opened", "Connection closed", "Query timeout"],
    "cache": ["Cache hit", "Cache miss", "Cache cleared", "Cache updated"],
    "payment": ["Payment initiated", "Payment completed", "Payment failed", "Refund processed"]
}

def generer_log():
    """Genere un log applicatif"""
    module = random.choice(MODULES)
    niveau = random.choice(NIVEAUX_LOG)
    message = random.choice(MESSAGES[module])
    
    return {
        "timestamp": datetime.now().isoformat(),
        "level": niveau,
        "module": module,
        "message": message,
        "request_id": f"req-{random.randint(10000, 99999)}",
        "user_id": f"user-{random.randint(1, 100)}",
        "duration_ms": random.randint(1, 500)
    }

In [ ]:
# Envoyer des logs
print("Envoi de logs...")
print("=" * 60)

for i in range(15):
    log = generer_log()
    
    # Utiliser le niveau comme cle
    future = producer.send(
        topic="logs-application",
        key=log["level"],
        value=log
    )
    
    result = future.get(timeout=10)
    print(f"[{log['level']:7}] {log['module']:10} | {log['message']}")

producer.flush()
print("=" * 60)

## 9. Producer de metriques

In [ ]:
def generer_metriques():
    """Genere des metriques systeme"""
    return {
        "timestamp": datetime.now().isoformat(),
        "host": f"server-{random.randint(1, 5):02d}",
        "metrics": {
            "cpu_percent": round(random.uniform(10, 95), 2),
            "memory_percent": round(random.uniform(30, 90), 2),
            "disk_percent": round(random.uniform(20, 85), 2),
            "network_in_mbps": round(random.uniform(10, 500), 2),
            "network_out_mbps": round(random.uniform(5, 200), 2),
            "requests_per_sec": random.randint(100, 5000),
            "response_time_ms": round(random.uniform(5, 100), 2)
        }
    }

# Envoyer des metriques
print("Envoi de metriques...")
for i in range(10):
    metric = generer_metriques()
    
    producer.send(
        topic="metrics",
        key=metric["host"],
        value=metric
    )
    
    print(f"{metric['host']} | CPU: {metric['metrics']['cpu_percent']}% | "
          f"MEM: {metric['metrics']['memory_percent']}%")

producer.flush()

In [ ]:
# Fermer le producer
producer.close()
print("Producer ferme")

---

## Exercice

**Objectif** : Creer un producer personnalise

**Consigne** :
1. Creez un topic "exercice-events"
2. Creez un generateur d'evenements utilisateur (click, view, purchase)
3. Envoyez 20 evenements avec une simulation de flux

A vous de jouer :

In [ ]:
# TODO: Creer le topic

In [ ]:
# TODO: Creer le generateur d'evenements
def generer_event():
    pass  # A completer

In [ ]:
# TODO: Envoyer les evenements

---

## Resume

Dans ce notebook, vous avez appris :
- Comment configurer un **Producer Kafka** robuste
- Comment **serialiser en JSON**
- Comment utiliser les **cles de partitionnement**
- Comment **envoyer en batch**
- Comment **simuler un flux** en temps reel
- Comment utiliser les **callbacks**

### Prochaine etape
Dans le prochain notebook, nous apprendrons a consommer les messages avec Spark.